In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Bladder Cancer Core Hub Gene PPI Network Analysis
Screen True Top 10 Core Genes via Multi-Metric Composite Score
(Pure screening version, no plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input bladder cancer gene list ====================
print("="*70)
print("Bladder Cancer Core Hub Gene Screening and Analysis (Screening Only)")
print("="*70)

# Bladder cancer related gene list
bladder_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "TP53", "PALB2", "CHEK2", "EGFR", "BRIP1", "MSH6", "MSH2",
    "MLH1", "CDH1", "APC", "PTEN", "C11orf65", "BARD1", "NF1", "RB1", "ERBB2", "PMS2",
    "CDKN2A", "POLD1", "KRAS", "POLE", "PIK3CA", "AXIN2", "DICER1", "FGFR3", "BRAF",
    "TSC1", "TSC2", "ALK", "CTNNB1", "RAD50", "MET", "STK11", "RET", "NBN", "HRAS",
    "TERT", "SMAD4", "MUTYH", "MSH3", "RAD51D", "PTCH1", "RAD51C", "SMARCA4", "BLM",
    "BAP1", "AKT1", "MRE11", "CTNNA1", "CDK4", "EPCAM", "KIT", "CDKN1B", "CCND1",
    "ARID1A", "PDGFRA", "ERCC2", "BMPR1A", "ESR1", "MYC", "FLCN", "MEN1", "SDHA",
    "FANCC", "RUNX1", "ERBB3", "FGFR2", "FH", "CD274", "AR", "HOXB13", "LZTR1",
    "MTOR", "VHL", "SDHB", "CDKN1A", "MLH3", "NRAS", "TUG1", "MDM2", "STAT3", "HIF1A",
    "NTHL1", "ATR", "VEGFA", "NF2", "POT1", "TGFBR2", "XRCC2", "SMARCB1", "SUFU",
    "TP63", "SOD2", "IL6", "RAD51", "KDM6A", "FGFR1", "BAX", "SDHD", "PDCD1", "BCL2",
    "SRC", "CASP8", "FOXO3", "MUC16", "BIRC5", "DNMT3A", "CDC73", "EP300", "NOTCH1",
    "CHEK1", "PRKAR1A", "SDHC", "EZH2", "CCNE1", "TGFB1", "GREM1", "ERBB4", "AOPEP",
    "PTGS2", "LRRC56", "MUC1", "ITGB1", "GATA3", "TMEM127", "FANCM", "MAP2K1",
    "RECQL4", "FANCA", "AURKA", "PIK3R1", "ERCC5", "WT1", "CDK6", "BUB1B", "XPC",
    "CD44", "PPARG", "PMS1", "TINCR", "NFE2L2", "CASP3", "RASSF1", "FANCD2", "MMP9",
    "CDKN2B", "DLC1", "MMP2", "GSTM1", "DHFR", "ERCC1", "MAPK1", "XRCC1", "RNF43",
    "XRCC3", "FBXW7", "GSTP1", "ERCC4", "FASLG", "SMARCE1", "TNFSF10", "SDHAF2",
    "KRT20", "ABCB1", "KDR", "CXCR4", "IL1B", "RAF1", "IGF2", "IQANK1", "MITF",
    "CDK12", "TNF", "NTRK1", "IGF1R", "CTLA4", "MAX", "JUN", "SMARCA5", "GLI1",
    "STAT1", "RAD51B", "PHOX2B", "FAS", "SPOP", "SMAD3", "DNMT1", "IDH1", "IL2",
    "AIP", "FOXO1", "PSCA", "KMT2D", "MGMT", "BLACAT1", "CHRNA3", "CREBBP", "RAD54L",
    "ABCG2", "GALNT12", "SMAD2", "TGFBR1", "PIK3CG", "MKI67", "CXCL8", "PARP1",
    "PRKDC", "ABCC1", "PPM1D", "GNAS", "EGF", "KRT7", "WRN", "MNX1", "ZKSCAN1",
    "FHIT", "NAT2", "PTK2", "SOX2", "CDKN1C", "CCL2", "BCL2L1", "ROBO1", "CTAG1B",
    "SMO", "NFKB1", "NRG1", "JAK2", "RNASEL", "HNF1B", "PTPN11", "GSTT1", "E2F1",
    "TMEM238L", "CD82", "WWOX", "TYMS", "IRF1", "MMP1", "IFNG", "FOXA1", "TFF1",
    "NFATC1", "MT-CYB", "NQO1", "UVRAG", "TFDP1", "ZFR", "MTHFR", "TLR4", "TWIST1",
    "KLK3", "DOCK1", "IGF1", "ERCC6", "FANCE", "CYP1A1", "FANCG", "IL10", "VANGL1",
    "FLT1", "RHOA", "ROS1", "XPA", "FANCL", "TP73", "GATA2", "XIAP", "CASP9",
    "SNAI1", "MCL1", "NPM1", "YAP1", "TOP2A", "CDK2", "ZEB2", "TLR2", "VEGFC",
    "RECQL", "IDH2", "RREB1", "GSK3B", "IL1RN", "PALLD", "SMAD7", "PRKN", "MAP2K2",
    "IGFBP3", "CDK1", "ABRAXAS1", "ERCC3", "RBPMS", "RARB", "FLT4", "FN1", "METTL3",
    "CTNNA3", "KLF4", "SHC1", "ELAC2", "TYMP", "FANCF", "FGFR4", "LRP1B", "CCND2",
    "KLF6", "RELA", "PCNA", "HNF1A", "CYP17A1", "NKX2-1", "SOX9", "SHH", "MSR1",
    "RBFOX2", "WEE1", "ERG", "AXIN1", "APEX1", "BUB1", "PRKCA", "ENG", "SPINK1",
    "CAV1", "MAP3K1", "BRINP1", "HBEGF", "AXL", "GPC3", "TCF7L2", "EPHB2", "FANCI",
    "PSMA3", "LMNA", "MMP14", "CXCL12", "OGG1", "HMMR", "FOLH1", "ACTA2", "CCND3",
    "HGF", "VDR", "AKT3", "BLCAP", "AKT2", "ADAR", "WNT3", "TNFRSF10B", "CYP19A1",
    "PDGFRB", "MMP7", "TBX3", "CYP1B1", "DCC", "SEPTIN9", "NTRK3", "LEFTY2", "PLAUR",
    "VIM", "SLC2A1", "LONP2", "RHBDF2", "NME1", "SETD2", "NTRK2", "PGR", "PLAU",
    "TMPRSS2", "HSP90AA1", "CEBPA", "KEAP1", "HSPB1", "KRT19", "AGO2", "CASZ1",
    "ESR2", "GSTM3", "PLVAP", "RUNX3", "HDAC1", "SERPINE1", "PPP2R2A", "CTAG2",
    "ZEB1", "CASR", "NFKBIA", "FGF2", "ATRX", "CBL", "UBAC2", "BMP4", "PROM1",
    "NOTCH3", "CA9", "CYP24A1", "FOXH1", "ITCH", "DLEC1", "SOX4", "BCAR1", "MTA1",
    "HAND2", "DDB2", "SYNE3", "MUC6", "UPK2", "NOTCH2", "TIMP1", "SP1", "SLX4",
    "SRD5A2", "FLNA", "FOXP3", "KMT2A", "MYH11", "MDM4", "CLPTM1L", "CDH2", "DNMT3B",
    "IL4", "IGF2R", "TGFA", "TAF4B", "DPYD", "CFTR", "SOCS1", "MAP2K4", "GJA1",
    "CSF3", "PBRM1", "STAG2", "ISL1", "ANXA2", "BMI1", "MYLK", "MYLK3", "CDX2",
    "EGR1", "EFEMP1", "POLK", "NAT1", "TIMP2", "HSPA5", "PHB1", "ERVH48-1", "NCOA3",
    "MESP1", "CBS", "DAPK1", "NCOR1", "CTSD", "FGF3", "AREG", "BRD4", "MXI1",
    "MUC2", "SPARC", "BIRC3", "MAPK14", "KLLN", "EPHA2", "HAND1", "CEACAM5", "ABL1",
    "ABCC4", "KMT2C", "PIK3R2", "ZNF224", "SNAI2", "CYP3A4", "PLK1", "JAK1",
    "S100A4", "TNFRSF10A", "IL1A", "MTR", "KLF5", "ETS1"
]

print(f"Number of input genes: {len(bladder_cancer_genes)}")
print(f"First 10 genes: {bladder_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI data from STRING database ====================
print("\n" + "="*70)
print("Stage 1: Retrieve PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Obtained {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Build NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(bladder_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions obtained: {len(interactions)}")

# Build network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate network topology metrics ====================
print("\n" + "="*70)
print("Stage 2: Calculate Network Topology Metrics")
print("="*70)

# Basic degree
degrees = dict(G.degree())
print("✓ Degree calculated")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree Centrality calculated")

# Betweenness centrality
print("Calculating Betweenness Centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness Centrality calculated (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness Centrality calculated")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector Centrality calculated")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector Centrality calculated (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculated")

# ==================== 4. Multi-metric composite scoring to screen core genes ====================
print("\n" + "="*70)
print("Stage 3: Multi-Metric Composite Scoring to Screen Core Hub Genes")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for metrics...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Bladder Cancer Top 20 Candidate Core Genes (sorted by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical analysis to determine core gene threshold ====================
print("\n" + "="*70)
print("Stage 4: Statistical Analysis to Determine Core Gene Threshold")
print("="*70)

# Statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite Score Statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Std Dev: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  3rd Quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite Score > {threshold:.2f})")

# Screen core genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nScreened {len(core_genes)} core genes by statistical threshold")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 core genes, taking top 10 by Composite Score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Less than 10 core genes, taking top 10 by Composite Score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract core subnetwork ====================
print("\n" + "="*70)
print("Stage 5: Construct Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing these core genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core Subnetwork Statistics:")
print(f"  Nodes: {subG.number_of_nodes()}")
print(f"  Edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export results ====================
print("\n" + "="*70)
print("Stage 6: Export Analysis Results")
print("="*70)

# Save core gene details
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('bladder_cancer_core_hub_genes.csv', index=False)
print("✓ Core Hub Gene details saved: bladder_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('bladder_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: bladder_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'bladder_cancer_core_network.graphml')
print("✓ Core network saved: bladder_cancer_core_network.graphml")

# ==================== 8. Final summary ====================
print("\n" + "="*70)
print("🎯 Bladder Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Top 10 Core Hub Genes (sorted by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("Generated files:")
print("  1. bladder_cancer_core_hub_genes.csv - Core gene detailed data")
print("  2. bladder_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. bladder_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Breast Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Breast Cancer Gene List ====================
print("="*70)
print("Breast Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Breast cancer-associated gene list
breast_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "PALB2", "BRIP1", "CHEK2", "BARD1", "CDH1", "C11orf65", "TP53",
    "MSH6", "MSH2", "MLH1", "ERBB2", "EGFR", "PTEN", "APC", "PMS2", "RAD51D", "RAD51C",
    "RAD50", "NF1", "ESR1", "POLD1", "PIK3CA", "POLE", "NBN", "STK11", "DICER1", "CDKN2A",
    "CTNNA1", "AKT1", "MUTYH", "AXIN2", "RET", "MET", "BRAF", "KRAS", "SMAD4", "RB1",
    "BLM", "TSC2", "SMARCA4", "ALK", "MRE11", "CDK4", "PTCH1", "EPCAM", "MSH3", "CTNNB1",
    "BAP1", "KIT", "FANCC", "BMPR1A", "CCND1", "TSC1", "TERT", "MYC", "MEN1", "FGFR2",
    "AR", "RAD51", "SDHA", "MTOR", "PDGFRA", "CDKN1B", "FH", "SDHB", "HRAS", "ATR",
    "MDM2", "RUNX1", "HIF1A", "HOXB13", "XRCC2", "FGFR3", "NTHL1", "FLCN", "TGFBR2", "VHL",
    "FANCM", "CASP8", "STAT3", "FGFR1", "AOPEP", "CDC73", "NRAS", "ERBB3", "SDHC", "FANCD2",
    "SDHD", "ERCC2", "NF2", "PPM1D", "CD274", "CHEK1", "IL6", "POT1", "ARID1A", "RAD54L",
    "LZTR1", "MLH3", "RAD51B", "SRC", "ERBB4", "RECQL4", "PMS1", "EP300", "GATA3", "TFF1",
    "SMARCB1", "VEGFA", "PRKAR1A", "CCNE1", "XRCC3", "FANCA", "CDK12", "MUC16", "CDKN2B", "PIK3R1",
    "MAP2K1", "JUN", "EZH2", "BCL2", "FBXW7", "ABCC1", "MT-CYB", "MUC1", "WT1", "NOTCH1",
    "CYP17A1", "TMEM127", "FOXA1", "RECQL", "TUG1", "BAX", "NTRK1", "ABRAXAS1", "CYP19A1", "GREM1",
    "MAP3K1", "SUFU", "TGFB1", "HMMR", "STAT1", "CDK6", "SDHAF2", "BUB1B", "MITF", "IL2",
    "PGR", "SHC1", "FANCE", "ABCG2", "PRKDC", "SLX4", "DCTN5", "CDKN1A", "CD44", "FANCG",
    "PTGS2", "WRN", "RNF43", "KMT2D", "CXCR4", "SMAD3", "FANCF", "MAX", "IL1B", "BIRC5",
    "IGF1R", "CREBBP", "FANCL", "NTRK3", "ERCC5", "AURKA", "PHB1", "ABCB1", "PDCD1", "RB1CC1",
    "ERCC4", "FHIT", "MTA1", "HNF1A", "BCAR1", "ERCC1", "CASP3", "BRMS1", "RINT1", "PPARG",
    "SMO", "MAPK1", "PARP1", "GALNT12", "XPC", "NTRK2", "JAK2", "AIP", "NCOR1", "SMARCE1",
    "TP63", "WEE1", "TGFBR1", "FANCI", "SEPTIN9", "MMP2", "PTK6", "MMP9", "ATRX", "NFE2L2",
    "PTK2", "XRCC1", "AGO2", "IGF1", "PHOX2B", "SLC67A1", "TNFSF10", "MAP2K4", "FASLG", "NRG1",
    "KDR", "SOD2", "LRP1B", "KDM6A", "ESR2", "MKI67", "GSTP1", "RASSF1", "FOXO1", "DLC1",
    "ITCH", "BLACAT1", "DHFR", "IGF2", "KMT2C", "TSG101", "PTPN11", "PBRM1", "CDKN1C", "ERCC3",
    "KLLN", "RAF1", "FAS", "EGF", "SOX2", "GLI1", "TNF", "CDC25B", "IL1RN", "PIK3CG",
    "KMT2A", "SMAD2", "CTLA4", "TWIST1", "DNMT1", "SNAI1", "DNMT3A", "NCOA3", "MGMT", "PELP1",
    "CKS1B", "TINCR", "NFKB1", "CDC6", "EXT2", "IL7R", "CDK2", "MTHFR", "FLT1", "LMNA",
    "SOX9", "GSK3B", "IQANK1", "GSTM1", "FOXO3", "TBX3", "NEK2", "XPA", "IDH1", "BCL2L1",
    "NQO2", "ABCA1", "GPC3", "PHB2", "E2F1", "AXL", "UIMC1", "PLAT", "IGFBP3", "YAP1",
    "HEATR6", "CXCL12", "GATA2", "CYP1A1", "DDB2", "FGFR4", "CXCL8", "NEK9", "PTCH2", "ZNF276",
    "AKT2", "IRF1", "WWOX", "GNAS", "TOP2A", "SMAD7", "TCF7L2", "VRK1", "ZNF462", "TMEM132E",
    "CASR", "RELA", "ACVR1B", "CDK1", "CCL2", "SPOP", "FAT4", "PALLD", "RHOA", "AKT3",
    "ROS1", "CCND2", "SPEN", "TYMS", "IDH2", "ZEB2", "CBFB", "CYP1B1", "ATRIP", "CTAG1B",
    "CASP9", "EXOC2", "KLC1", "STAG2", "KRT7", "CAV1", "IFNG", "CEBPA", "KIF4A", "KLK3",
    "CTSD", "MCL1", "ENG", "HDAC1", "ZEB1", "UFC1", "PRLR", "BACH1", "XIAP", "RNASEL",
    "LEP", "ITGB1", "AXIN1", "VEGFC", "IL10", "VDR", "HELZ", "KRT20", "JAK1", "PDGFRB",
    "SLC2A1", "FOXP3", "FLT4", "ALDH1A1", "ADIPOQ", "PCNA", "KEAP1", "PRKN", "SETD2", "PLAU",
    "LRBA", "MMP14", "MMP1", "LDLR", "CBL", "PRKCA", "DROSHA", "RARB", "RHBDF2", "ACTA2",
    "KLF4", "NQO1", "PROM1", "PBOV1", "FN1", "PCGF2", "BUB1", "ABCC6", "NFKBIA", "HSPA5",
    "GPER1", "PLAUR", "TP73", "TLR4", "CACNA2D1", "NKX2-1", "CD82", "ESS2", "SERPINE1", "SPINK1",
    "IRAK4", "SP1", "TRAF5", "ANXA2", "CCNB1", "CASP10", "PPP2R2A", "MAP2K2", "PTHLH", "DLEC1",
    "DDX21", "NPM1", "SLC26A4", "MAPK3", "RIPK1", "NAT2", "ACTC1", "BCAR3", "ATP7B", "HSP90AA1",
    "TUBA4B", "BSCL2", "PAH", "CYP24A1", "BMP6", "DCC", "AHR", "GSTT1", "KRT19", "KLF6",
    "RARA", "GATD3", "KDM1A", "HGF", "LCP1", "FOLH1", "TATDN1", "TGFA", "PSTPIP1", "VIM",
    "SNAI2", "FGF2", "IRS1", "KISS1", "SIRT1", "FBXO32", "FOXM1", "ELAC2", "TNFRSF10B", "TPM1",
    "WRAP53", "ERCC6", "CREB1", "APEX1", "NAA25", "CCND3", "SNCG", "CTCF", "CSF3", "NOTCH2",
    "NME1", "MSR1", "CD24", "FASN", "AREG", "BMI1", "HNF1B", "GNG3", "OMA1", "KIF1B",
    "NSD1", "CDKN3", "NOTCH3", "STAT5A", "SRD5A2", "BIRC3", "SPAAR", "TIMP1", "TOP1", "HSPB1",
    "BECN1", "PSCA", "ABL1", "EXT1", "RHOT1", "SPP1", "COL3A1", "CNOT2", "CEACAM5", "CA9",
    "CDH2", "SOCS1", "H2AX", "ETS1", "FANCB", "ZNF226", "SOS1", "BAK1", "OGG1", "MMP7",
    "TRIM24", "MDM4", "IGF2R", "TGFB2", "FMN1", "KRT5", "RBBP8", "COMT", "METTL3", "MAPK8",
    "ZFHX3", "FGF3", "ETV6", "RP1", "INSR", "PRC1", "SRA1", "RRAS2", "RPS20", "SOX4",
    "TNFSF11", "WNT5A", "ELAVL1", "GREB1", "PIK3CD", "DDR1", "DPYD", "PRF1", "PTGFR", "GRB2",
    "CYP3A4", "GSTM3", "MC1R", "CSF1R", "ZBED4", "BCL2L11", "FOS", "AURKB", "TYMP", "ERG",
    "TIMP2", "HMGA2", "RNF10", "FADD", "MAPK14", "MAPK10", "EPHB2", "MUC6", "RUNX3", "CFTR",
    "FBN1", "ABCC4", "EPHB4", "TOE1", "BIVM-ERCC5", "DIS3L2", "TLR2", "RAC1", "LEF1", "HDAC4",
    "YBX1", "IL4", "PLK1", "CDC25A", "PIK3R2", "DNMT3B", "BCAS1", "RHOBTB2", "PIK3CB", "DSG2",
    "TIMP3", "ODC1", "RPS6KB1", "PRL", "ENO1", "TET2", "DSP", "NT5E", "ACTB", "SHH",
    "PPP2R1B", "PML", "BMP2", "TMEM91", "APOB", "PPM1L", "MAP3K6", "XRCC6", "SCGB2A2", "PDGFB",
    "PKM", "BRD4", "RABL3", "DKK1", "S100A4", "PDPN", "CCNA2", "BLTP2", "HMGB1", "MYH11",
    "BMP4", "EPHA2", "C10orf143", "MTDH", "GDF15", "CCL5", "PAK1", "KRT8", "CYLD", "KRT18",
    "ABCC2", "CSF1", "KDM4B", "STAT5B", "TMPRSS2", "ALDH2", "SLC19A1", "BAD", "MYB", "EGR1",
    "KLF5", "IL2RA", "ETV4", "NCOR2", "HSD17B1", "HDAC6", "FLT3", "IRS2", "CYP2D6", "MED12",
    "CCN2", "CFLAR", "GNRH1", "LGALS3", "LDHA", "STAT6", "SPARC", "ACD", "LEPR", "PRMT5",
    "CYCS", "POSTN", "MBD4", "TUBB3", "SULT1A1", "CLPTM1L", "JAG1", "PTPN3", "BSG", "PCSK9",
    "TNFRSF10A", "XBP1", "VEGFD", "CDH3", "TOX3", "SKP2", "KLK10", "YY1", "CDX2", "GADD45A",
    "KCNH2", "ITGA2", "TGFBR3", "BCL6", "SQSTM1", "CSF2", "SOCS3", "INS", "POLK", "PTK2B",
    "ANKRD30A", "ICAM1", "ID1", "THBS1", "ITGA5", "MYCN", "MXI1", "KITLG", "FGF7", "FGF4",
    "CDC25C", "BCL10", "TSHR", "NCOA1", "POU5F1", "MELK", "GNRHR", "FGF10", "MMP11", "PRKACA",
    "MTR", "MED1", "PAK4", "ING1", "CASP7", "KRT14", "WNT1", "IL1A", "MUC5AC", "PAX5",
    "PLCG1", "IGF2BP3", "HBEGF", "GJA1", "PPP2R1A", "RUNX2", "PRKCD", "LIG4", "CDC42", "BID",
    "SLC34A2", "BCAS3", "EIF4EBP1", "SERPINB5", "PLK2", "FGF8", "EPHA3", "SMARCA2", "CCR7", "NANOG",
    "TNFRSF11A", "LCN2", "PTPRC", "HSPA4", "APOD", "XRCC5", "RHOC", "MUC4", "E2F3", "ADAM17",
    "IFI27", "SF3B1", "CXCL1", "CD36", "HDAC2", "TP53BP1", "MYLK", "CTTN", "EFNA1", "NRP1",
    "IKBKB", "EPAS1", "FSCN1", "MME", "ITGA6", "GEN1", "TSPAN31", "TRPS1", "KDM5B", "COL1A1",
    "RRAS", "CLU", "BMP7", "SYK", "HDAC9", "ADAR", "CEBPB", "MMP13", "IDO1", "ZNF217",
    "NOTCH4", "MAGEA3", "CYP1A2", "CALR", "MMP3", "NR3C1", "CSK", "HFE", "HMOX1"
]

print(f"Number of input genes: {len(breast_cancer_genes)}")
print(f"First 10 genes: {breast_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(breast_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Breast Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('breast_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: breast_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('breast_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: breast_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'breast_cancer_core_network.graphml')
print("✓ Core network saved: breast_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Breast Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. breast_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. breast_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. breast_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Kidney Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Kidney Cancer Gene List ====================
print("="*70)
print("Kidney Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Kidney cancer-associated gene list
kidney_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "PKD1", "PKHD1", "PALB2", "CHEK2", "TP53", "BRIP1", "MSH6",
    "MSH2", "MLH1", "CDH1", "EGFR", "APC", "BARD1", "TSC2", "MET", "PTEN", "C11orf65",
    "PMS2", "NF1", "RET", "DICER1", "PKD2", "POLD1", "POLE", "AXIN2", "BRAF", "ERBB2",
    "TSC1", "VHL", "FLCN", "CDKN2A", "ALK", "RAD50", "STK11", "SMARCA4", "BAP1", "PIK3CA",
    "SMAD4", "PTCH1", "RB1", "RAD51C", "NBN", "MUTYH", "CTNNB1", "RAD51D", "HNF1B", "MSH3",
    "BLM", "WT1", "AKT1", "KRAS", "KIT", "FH", "NOTCH2", "CTNNA1", "UMOD", "EPCAM",
    "MTOR", "PDGFRA", "CDK4", "BMPR1A", "MEN1", "MUC1", "MRE11", "TERT", "SDHA", "RUNX1",
    "SDHB", "CDKN1B", "ESR1", "FANCC", "CCND1", "MYC", "FGFR3", "FGFR2", "HRAS", "HNF1A",
    "HOXB13", "SMARCB1", "LZTR1", "STAT3", "HIF1A", "GREM1", "PAX2", "MLH3", "IL6", "CD274",
    "ATR", "NTHL1", "TGFB1", "SDHD", "GATA3", "GANAB", "AR", "CDC73", "NF2", "MDM2",
    "SDHC", "TUG1", "VEGFA", "TGFBR2", "POT1", "NRAS", "REN", "PRKAR1A", "FGFR1", "SUFU",
    "XRCC2", "LRP5", "TNF", "CASR", "NOTCH1", "RAD51", "TMEM127", "ARID1A", "MITF", "CTLA4",
    "CEP290", "EP300", "AOPEP", "BAX", "ACE", "PDCD1", "NEK8", "SRC", "COL4A4", "ERCC2",
    "BCL2", "SDHAF2", "CASP8", "FANCD2", "CHEK1", "FANCM", "ABCB1", "NPHP3", "COL4A5", "RECQL4",
    "IGF2", "BUB1B", "ERBB3", "APOL1", "NPHP4", "COL4A3", "PPARG", "IL1B", "EZH2", "MYH9",
    "MAX", "NFE2L2", "CXCR4", "MUC16", "LCN2", "MAP2K1", "PTGS2", "PRKCSH", "BMP4", "NPHS1",
    "NPHP1", "EYA1", "PIK3R1", "ERBB4", "FANCA", "NTRK1", "PHOX2B", "PMS1", "DLC1", "GLIS2",
    "PAX8", "CASP3", "SEC63", "IFNG", "JAG1", "SMARCE1", "CDKN1C", "CDKN1A", "CFH", "KDR",
    "MAPK1", "CYP24A1", "BMP7", "CD44", "GPC3", "SMAD3", "VDR", "INS", "FASLG", "FBXW7",
    "NPHS2", "RNF43", "PBX1", "CCL2", "IGF1R", "INVS", "WRN", "CUBN", "JUN", "GATA2",
    "FN1", "CRB2", "IDH1", "IL2", "FLT1", "BIRC5", "IL10", "GLI1", "FAS", "ACTN4",
    "SPOP", "AURKA", "TMEM67", "CDKN2B", "DHFR", "HNF4A", "CCNE1", "RAF1", "IFT140", "CDK6",
    "KMT2D", "TGFBR1", "AGTR1", "AIP", "GNAS", "FGF23", "PBRM1", "ERCC4", "ABCG2", "PIK3CG",
    "ACTA2", "ERCC5", "PTPN11", "SOX9", "SMAD2", "EGF", "SDCCAG8", "JAK2", "RAD51B", "RASSF1",
    "MMP9", "CDK12", "DNMT1", "GALNT12", "FOXO1", "STAT1", "MMP2", "IGF1", "CFTR", "SOD2",
    "DNAJB11", "SALL1", "SIX1", "ERCC1", "TNFSF10", "DZIP1L", "ADIPOQ", "FAT4", "MTHFR", "RPGRIP1L",
    "AGT", "DNMT3A", "INF2", "PPM1D", "PARP1", "SOX2", "SMO", "PTK2", "WNT4", "WNT9B",
    "FHIT", "TLR4", "SETD2", "XPC", "FANCG", "FANCE", "GSTP1", "PLCE1", "AQP2", "CXCL8",
    "TCF7L2", "CREBBP", "APOA1", "FANCL", "SLC12A1", "COL4A1", "HGF", "MKS1", "ALPL", "GLA",
    "TRPC6", "GSK3B", "SMARCAL1", "ABCC1", "FOXP3", "OFD1", "BICC1", "CC2D2A", "ITGB1", "SPP1",
    "NFKB1", "SIX2", "IL1RN", "FANCF", "RAD54L", "XRCC3", "HAVCR1", "ENG", "DSTYK", "SHH",
    "CST3", "FANCI", "LAMB2", "SMAD7", "ROS1", "KDM6A", "FOXO3", "IDH2", "RECQL", "PALLD",
    "WDR19", "PDGFRB", "CD2AP", "RHOA", "OGG1", "MGMT", "IRF1", "ALB", "TFF1", "SEC61A1",
    "ROBO2", "B2M", "NTRK3", "TP63", "TTC21B", "RNASEL", "CYP17A1", "SLC12A3", "XRCC1", "GSTM1",
    "NLRP3", "KRT7", "DIS3L2", "MME", "YAP1", "NRG1", "AXL", "MT-CYB", "PROM1", "PRKDC",
    "CD46", "CXCL12", "PRKN", "KEAP1", "LMX1B", "SNAI1", "CTAG1B", "ALG8", "ERCC6", "FOLH1",
    "MAP3K1", "AKT2", "FGFR4", "ATRX", "CRP", "BUB1", "IGFBP3", "NKX2-1", "RELA", "FREM2",
    "ERCC3", "SOX17", "ELAC2", "SLC2A1", "CLCNKB", "CEBPA", "NTRK2", "AGXT", "SERPINE1", "AKT3",
    "XPA", "ROBO1", "TLR2", "RARB", "FOXA1", "C3", "AXIN1", "EPO", "BCL2L1", "CCN2",
    "ZEB2", "FBN1", "CYP3A4", "SHC1", "KCNJ1", "DLEC1", "SLX4", "MKI67", "CYS1", "KLF6",
    "XIAP", "DYNC2H1", "SLC67A1", "DGKE", "HMOX1", "BCAR1", "TMPRSS2", "ABRAXAS1", "KRT20", "ITGA8",
    "NFKBIA", "AGO2", "ITCH", "CLCN5", "ITGA3", "MYO1E", "FOXC1", "DCC", "YWHAE", "CBL",
    "LAMA5", "CAV1", "TFE3", "FGF2", "FRAS1", "TBX18", "CLCNKA", "VEGFC", "GDNF", "METTL3",
    "ABCC2", "AVPR2", "ANKS6", "GNA11", "INSR", "NRIP1", "FLT4", "BSND", "SHOC2", "LEP",
    "LMNA", "WWOX", "SIX5", "KLF4", "HMMR", "PGR", "CDK2", "HSPA5", "CASP9", "ABCC4",
    "NPM1", "LRP1B", "MMP1", "TCTN2", "CYP1A1", "TYMS", "WNT5A", "MCL1", "GLI3", "MYCN",
    "ETV4", "IL2RA", "TMEM216", "PCNA", "CUL3", "HDAC1", "TWIST1", "LRP2", "PRKCA", "TFAP2A",
    "ERG", "COQ8B", "CYP19A1", "CEP164", "CBS", "FAN1", "CFI", "SOCS1", "MAP2K2", "ZNF423",
    "E2F1", "SLC4A1", "CCND2", "KLLN", "TRIM28", "NQO1", "KIF4A", "MAP2K4", "DDB2", "TMEM231",
    "XPNPEP3", "NEK9", "SIRT1", "PKD2L1", "PIK3CD", "GREB1L", "OCRL", "SLC7A9", "PSCA", "MAPK3",
    "KLK3", "VIM", "TIMP2", "RHBDF2", "AHI1", "CA9", "TGFB2", "UPK3A", "NSD1", "NOTCH3",
    "GRHPR", "PTHLH", "RARA", "ETS1", "THBD", "MTA1", "ANXA2", "CLDN16", "JAK1", "CYP1B1",
    "EPHB2", "DCDC2", "PTH", "BLACAT1", "MSR1", "HOGA1", "CASP10", "CDK1", "SPINK1", "RPS20",
    "SEPTIN9", "PDK1", "WNK1", "SLC34A1", "NOD2", "CALCA", "KMT2A", "MAPK8", "BCOR", "DROSHA",
    "PTPRO", "IL18", "ATP6V1B1", "BBS1", "HLA-B", "FLNA", "ALG9", "ABCA1", "BRD4", "ATP7B",
    "SLC4A4", "CYP3A5", "CEP83", "FANCB", "TOP2A", "TGFA", "PHB1", "HMGB1", "CD82", "WEE1",
    "CFHR5", "HLA-A", "ICAM1", "BMP2", "NOS3", "PLAUR", "SLC26A4", "KMT2C", "CREB1", "TNFSF11",
    "TP73", "TIMP1", "ARHGDIA", "BMP6", "MMP7", "HSP90AA1", "EPAS1", "ANLN", "FZD3", "TNFRSF10B",
    "PLAU", "LGALS3", "KDM1A", "IL4", "ZNF609", "IQCB1", "GSTM3", "ZEB1", "CPLANE1", "HFE",
    "PIK3CB", "DNMT3B", "TSHZ3", "GFRA1", "ALG5", "HPRT1", "BIRC3", "SRGAP1", "PCSK9", "TET2",
    "IRS1", "PDPN", "KCNQ1", "FGF20", "IFT172", "GDF15", "ABL1", "FGF10", "HLA-DRB1", "APOB",
    "EXT2", "ESR2", "NSD2", "DKK1", "CLU", "FOXM1", "MXI1", "HIPK3", "HMGA2", "MAPKBP1",
    "REST", "TRAP1", "MAPK10", "TNFRSF1A", "IL7R", "CD36", "LDLR", "DCTN5", "BBS2", "KIF1B",
    "STAT6", "HAVCR2", "ITGAM", "PDGFB", "SLC3A1", "COMT", "SP1", "PPP2R2A", "SOX4", "FGF3",
    "BBS9", "KRT19"
]

print(f"Number of input genes: {len(kidney_cancer_genes)}")
print(f"First 10 genes: {kidney_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(kidney_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Kidney Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('kidney_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: kidney_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('kidney_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: kidney_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'kidney_cancer_core_network.graphml')
print("✓ Core network saved: kidney_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Kidney Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. kidney_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. kidney_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. kidney_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Laryngeal Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Laryngeal Cancer Gene List ====================
print("="*70)
print("Laryngeal Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Laryngeal cancer-associated gene list
laryngeal_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "PALB2", "CHEK2", "TP53", "BRIP1", "MSH2", "MSH6", "MLH1",
    "CDH1", "EGFR", "APC", "BARD1", "PTEN", "C11orf65", "PMS2", "NF1", "POLD1", "POLE",
    "DICER1", "AXIN2", "ERBB2", "CDKN2A", "RAD50", "TSC2", "ALK", "STK11", "RET", "BRAF",
    "MET", "RB1", "RAD51D", "SMAD4", "NBN", "MUTYH", "SMARCA4", "RAD51C", "PTCH1", "MSH3",
    "PIK3CA", "BLM", "BAP1", "KRAS", "CTNNA1", "KIT", "EPCAM", "CTNNB1", "CDK4", "PDGFRA",
    "MRE11", "AKT1", "TSC1", "BMPR1A", "RUNX1", "CCND1", "TERT", "MEN1", "CDKN1B", "FLCN",
    "SDHA", "FANCC", "ESR1", "FH", "HOXB13", "MYC", "VHL", "HRAS", "LZTR1", "STAT3",
    "MLH3", "FGFR3", "SDHB", "MTOR", "CD274", "FGFR2", "NTHL1", "AR", "POT1", "HIF1A",
    "MDM2", "NRAS", "CDC73", "TGFBR2", "SMARCB1", "VEGFA", "XRCC2", "SDHC", "SDHD", "SUFU",
    "PRKAR1A", "BCL2", "RAD51", "ERCC2", "BAX", "IL6", "NOTCH1", "FGFR1", "ARID1A", "TMEM127",
    "CHEK1", "CASP8", "ERBB3", "BIRC5", "GREM1", "EP300", "MUC16", "FANCM", "PTGS2", "RECQL4",
    "TGFB1", "ERBB4", "PMS1", "SRC", "EZH2", "PDCD1", "SDHAF2", "CD44", "TUG1", "CCNE1",
    "FBXW7", "RNF43", "PHOX2B", "MAX", "AURKA", "FANCD2", "MITF", "MAP2K1", "WT1", "CDKN1A",
    "CXCR4", "CDKN2B", "SMARCE1", "ERCC5", "FANCA", "GALNT12", "MMP2", "CASP3", "DHFR", "IL1B",
    "NFE2L2", "MMP9", "PIK3R1", "XRCC1", "NTRK1", "MUC1", "PPARG", "MAPK1", "RAD51B", "IDH1",
    "CDK6", "ERCC1", "FASLG", "CDK12", "IGF1R", "PPM1D", "KDR", "MGMT", "WRN", "ABCB1",
    "SOX2", "ERCC4", "JUN", "GSTM1", "XPC", "RAF1", "PTK2", "XRCC3", "TNFSF10", "SOD2",
    "TNF", "GSTP1", "PTPN11", "CTLA4", "SMAD3", "FOXO1", "RASSF1", "JAK2", "MKI67", "RNASEL",
    "FAS", "GLI1", "SMAD2", "IGF2", "FHIT", "ABCG2", "ROS1", "GNAS", "GATA3", "DNMT1",
    "RECQL", "FLT1", "CREBBP", "TP63", "SMO", "PIK3CG", "IL2", "FANCG", "IFNG", "STAT1",
    "MT-CYB", "CTAG1B", "TNFRSF10B", "XPA", "DNMT3A", "PROM1", "CXCL8", "PARP1", "IDH2", "EGF",
    "FANCL", "KMT2D", "SOX9", "ABRAXAS1", "CYP1A1", "NFKB1", "ENG", "IRF1", "TCF7L2", "YAP1",
    "BCL2L1", "CDKN1C", "PRKDC", "MTHFR", "CDK2", "IL1RN", "NKX2-1", "PCNA", "VEGFC", "HNF1A",
    "IGF1", "KLF6", "GPC3", "ERCC3", "GSK3B", "CD82", "FANCF", "IL10", "SNAI1", "TYMS",
    "FANCI", "ERCC6", "CEBPA", "SMAD7", "RELA", "AXIN1", "TWIST1", "FLT4", "E2F1", "MCL1",
    "ITGB1", "NTRK2", "DLEC1", "MTA1", "BUB1", "BLACAT1", "CXCL12", "AKT2", "MMP1", "TP73",
    "KRT20", "GSTT1", "KEAP1", "AXL", "ZEB2", "XIAP", "RARB", "RHOA", "CCL2", "TLR4",
    "CASP9", "GSTM3", "WEE1", "NME1", "SLC2A1", "SLX4", "TLR2", "KLK3", "PGR", "MAP2K4",
    "METTL3", "ERG", "CDK1", "HDAC1", "TMPRSS2", "BMI1", "NPM1", "ATRX", "KRT7", "KLF4"
]

print(f"Number of input genes: {len(laryngeal_cancer_genes)}")
print(f"First 10 genes: {laryngeal_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(laryngeal_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Laryngeal Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('laryngeal_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: laryngeal_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('laryngeal_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: laryngeal_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'laryngeal_cancer_core_network.graphml')
print("✓ Core network saved: laryngeal_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Laryngeal Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. laryngeal_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. laryngeal_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. laryngeal_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Leukemia Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Leukemia Gene List ====================
print("="*70)
print("Leukemia Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Leukemia-associated gene list
leukemia_genes = [
    "CEBPA", "GATA2", "RUNX1", "FLT3", "TP53", "NPM1", "NF1", "CBL", "KMT2A", "KIT",
    "ABL1", "PTPN11", "TERT", "JAK2", "IKZF1", "BCR", "KRAS", "DNMT3A", "ETV6", "PML",
    "NRAS", "ATM", "NBN", "STAT3", "TET2", "BCL2", "CDKN2A", "MYC", "PAX5", "RARA",
    "ASXL1", "NOTCH1", "GATA1", "TAL1", "IDH1", "WT1", "BRAF", "MPL", "CSF3R", "CREBBP",
    "NUP214", "BAX", "TLX1", "STAT5B", "SF3B1", "MLLT10", "SRSF2", "SETBP1", "PTEN", "CCND1",
    "IDH2", "RUNX1T1", "MLLT3", "NUP98", "EZH2", "AKT1", "TCF3", "TLX3", "LIF", "CXCR4",
    "PBX1", "ZBTB16", "BTK", "PDGFRB", "JAK3", "SAMD9L", "SETD2", "HRAS", "MECOM", "JAK1",
    "FBXW7", "ERG", "BCL6", "AFF1", "CBFB", "MLF1", "PDGFRA", "MAP2K1", "DDX41", "RB1",
    "EGFR", "CHEK2", "BRCA2", "MCL1", "RAF1", "EP300", "CALR", "FGFR1", "BCL11B", "MLLT1",
    "CSF1R", "MPO", "BIRC3", "PICALM", "U2AF1", "IRF4", "AFDN", "NSD1", "SAMD9", "PIK3CA",
    "CDK6", "STAT5A", "KAT6A", "BCOR", "CRLF2", "CD274", "KMT2D", "MRTFA", "MYB", "PALB2",
    "TCL1A", "LYL1", "BAALC", "PIK3CD", "MYH11", "BRCA1", "PDCD1", "MLH1", "CARD11", "CTLA4",
    "MYD88", "CCND3", "XPO1", "STAT1", "IKZF3", "ALK", "IGH", "EPOR", "ERBB2", "EBF1",
    "IL7R", "SH2B3", "FASLG", "STIL", "DNMT1", "CTNNB1", "CDK4", "MSH2", "STAG2", "IRF8",
    "GFI1", "PHF6", "FAS", "DNTT", "HOXA9", "TAL2", "CSF3", "SYK", "ATRX", "TYK2",
    "KLF2", "SH3GL1", "MSH6", "RAD21", "TNFAIP3", "DEK", "TRB", "ABCB1", "DCK", "GATA3",
    "CEBPE", "ABL2", "PTK2B", "CASP3", "ZRSR2", "MDM2", "CD79B", "DDX3X", "SRC", "FGFR3",
    "KDM6A", "PIK3R1", "ERBB4", "ELANE", "RHOA", "NTRK3", "SOCS1", "CBFA2T3", "ADA", "SUZ12",
    "AKT3", "CUX1", "BCL10", "MET", "NOTCH2", "SMC3", "PLCG2", "CHIC2", "PIK3CG", "ARID1A",
    "PIM1", "LIFR", "IKZF2", "BCORL1", "CBLB", "BCL3", "INSL6", "KDM1A", "P2RY8", "MAP2K2",
    "PDCD1LG2", "CD33", "SBDS", "CD79A"
]

print(f"Number of input genes: {len(leukemia_genes)}")
print(f"First 10 genes: {leukemia_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(leukemia_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Leukemia (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('leukemia_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: leukemia_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('leukemia_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: leukemia_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'leukemia_core_network.graphml')
print("✓ Core network saved: leukemia_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Leukemia Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. leukemia_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. leukemia_all_genes_metrics.csv - Network metrics for all genes")
print("  3. leukemia_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Liver Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Liver Cancer Gene List ====================
print("="*70)
print("Liver Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Liver cancer-associated gene list
liver_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "TP53", "PALB2", "APC", "CHEK2", "BRIP1", "MSH2", "MSH6",
    "MLH1", "EGFR", "CDH1", "BARD1", "PKHD1", "PTEN", "PMS2", "C11orf65", "NF1", "TSC2",
    "MET", "ERBB2", "POLD1", "DICER1", "CDKN2A", "BRAF", "POLE", "AXIN2", "RAD50", "ALK",
    "CTNNB1", "STK11", "SMAD4", "RET", "PIK3CA", "PKD1", "KRAS", "NBN", "SMARCA4", "RB1",
    "BAP1", "KIT", "TSC1", "MUTYH", "PTCH1", "AKT1", "RAD51D", "RAD51C", "MSH3", "BLM",
    "PDGFRA", "EPCAM", "TERT", "CTNNA1", "CDK4", "FH", "CCND1", "BMPR1A", "MRE11", "RUNX1",
    "ESR1", "VHL", "HRAS", "MEN1", "CDKN1B", "FLCN", "MYC", "SDHA", "FGFR2", "FANCC",
    "MTOR", "STAT3", "FGFR3", "IL6", "NRAS", "SDHB", "LZTR1", "CASP8", "HOXB13", "AR",
    "HIF1A", "CD274", "NOTCH1", "TGFB1", "ARID1A", "PPARG", "MLH3", "ATR", "NF2", "NTHL1",
    "SMARCB1", "PRKCSH", "MDM2", "TGFBR2", "VEGFA", "POT1", "FGFR1", "LRP5", "SEC63", "PKD2",
    "HNF1A", "SDHD", "SRC", "BAX", "SDHC", "PRKAR1A", "HNF1B", "TNF", "CDC73", "MAP2K1",
    "XRCC2", "DLC1", "TUG1", "RAD51", "ERCC2", "BCL2", "FASLG", "MARS1", "PDCD1", "WT1",
    "CYP3A4", "GREM1", "SUFU", "ERBB4", "ABCB1", "MUC1", "CHEK1", "GANAB", "AOPEP", "EP300",
    "PTK2", "GPC3", "NFE2L2", "IL1B", "ERBB3", "HNF4A", "HFE", "EZH2", "IDH1", "NTRK1",
    "IGF1R", "MUC16", "CXCR4", "RECQL4", "FAS", "PNPLA3", "PTPN11", "TMEM127", "PIK3R1", "FANCM",
    "JAK2", "PTGS2", "RAF1", "AXIN1", "GNAS", "KDR", "FANCD2", "CTLA4", "TRMU", "IGF2",
    "ATP7B", "SERPINA1", "PMS1", "CD44", "CASP3", "BIRC5", "FBXW7", "SMAD3", "JUN", "KRT7",
    "CCNE1", "RNF43", "FANCA", "RINT1", "BUB1B", "IDH2", "MITF", "ADIPOQ", "CDKN1A", "STAT1",
    "ROS1", "SMAD7", "SDHAF2", "ABCC2", "SMO", "MAPK1", "SOD2", "FOXO1", "IL10", "MMP9",
    "GYS2", "IFNG", "INS", "AURKA", "ERCC1", "DHFR", "MMP2", "CEBPA", "TNFSF10", "NRG1",
    "CREBBP", "MAX", "CDK6", "SMAD2", "CDKN2B", "AIP", "SMARCE1", "ERCC5", "CCL2", "ALB",
    "CFTR", "LMNA", "ITCH", "ABCG2", "CDK12", "NBAS", "PHOX2B", "IGF1", "GLI1", "TLR4",
    "IL2", "GSTP1", "ERCC4", "RASSF1", "HGF", "FLT1", "ENG", "AXL", "RAD51B", "MTHFR",
    "VDR", "WRN", "KRT20", "CYP2A6", "SPOP", "PIK3CG", "NOTCH2", "EGF", "AFP", "GATA3",
    "NTRK3", "FGFR4", "GALNT12", "AKT2", "ALG8", "CYP2E1", "NFKB1", "IGF2R", "CDC25A", "AGO2",
    "IL1RN", "FOXA1", "CXCL8", "ABCC1", "GSTM1", "XPC", "NTRK2", "DNMT1", "GATA2", "PROM1",
    "KRT18", "GPT", "PPM1D", "JAG1", "CYP1A1", "SLC2A1", "PARP1", "FOXP3", "DNMT3A", "MGMT",
    "UGT1A1", "IRF1", "KMT2D", "ALDH2", "FANCI", "APOB", "TGFBR1", "PBRM1", "INSR", "YAP1",
    "CDKN1C", "FOXO3", "TYMS", "ABCA1", "PALLD", "MT-CYB", "GSK3B", "ABCB11", "CP", "SOX2",
    "BLACAT1", "CXCL12", "CYP1A2", "TCF7L2", "ACTA2", "PPARA", "CAV1", "XRCC3", "RAD54L", "ITGB1",
    "FHIT", "KLF6", "FANCG", "CBS", "BCL2L1", "GSTM3", "XIAP", "TLR2", "PRKDC", "XRCC1",
    "LDLR", "SNAI1", "HSPA5", "IGFBP3", "NR1H4", "CDKN3", "SHC1", "SLC17A5", "RNASEL", "PRMT5",
    "TYMP", "CTAG1B", "PRKN", "SOX9", "NFKBIA", "SIRT1", "SLCO1B1", "NPM1", "RELA", "FANCE",
    "FOXM1", "ERCC3", "RECQL", "MKI67", "CYP17A1", "PHKA2", "XPA", "MCL1", "CBL", "PCSK9",
    "FANCL", "CDK2", "FASN", "LEP", "TFF1", "CASR", "JAK1", "RHOA", "NKX2-1", "KEAP1",
    "FN1", "SHH", "E2F1", "PDGFRB", "LRP6", "TWIST1", "FBN1", "DPYD", "MTA1", "F2",
    "PYGL", "FANCF", "ERCC6", "IRS1", "NR1H3", "SERPINE1", "TINCR", "ZEB2", "KRT19", "ABCB4",
    "BMP6", "CASP9", "RARB", "MAP3K1", "RHBDF2", "ATRX", "ELAC2", "FLT4", "SREBF1", "SETD2",
    "ABCC6", "NQO1", "ARG1", "BUB1", "IQANK1", "CPT1A", "TIMP1", "HMOX1", "SOCS1", "ANXA2",
    "CD82", "ABRAXAS1", "KRT8", "CYP19A1", "HDAC1", "SLC25A13", "CDK1", "SPP1", "MMP1", "HADHA",
    "DDR2", "AKT3", "ADAR", "G6PC1", "GGT1", "XBP1", "CYP1B1", "ABCC4", "PGR", "SMARCA2",
    "PCNA", "SLC25A15", "TNFRSF10B", "TP63", "LRP1B", "CYP2C19", "SOCS3", "IL21R", "MAPK8", "TP73",
    "FABP1", "APOA1", "SLCO1B3", "MYCN", "CALCA", "CEACAM5", "METTL3", "SPINK1", "VEGFC", "EFNA1",
    "IL2RA", "SLX4", "DKK3", "SEPTIN9", "DLEC1", "MSR1", "IL4", "KDM6A", "POLG", "DDB2",
    "NAT2", "PHKG2", "HSP90AA1", "RARA", "APOE", "CRP", "TMPRSS2", "BIRC3", "TNFRSF1A", "HMMR",
    "NR1H2", "HAMP", "CREB1", "AHR", "ALDH1A1", "CYP2D6", "KLF4", "PRKCA", "LARS1", "MAPK3",
    "TOP2A", "PDGFRL", "HLA-DRB1", "MYLK", "PHB1", "DDIT3", "CALR", "FLT3", "ABL1", "ERG",
    "FADD", "VIM", "PLAU", "WWOX", "RXRA", "CYP24A1", "MMP7", "MAP2K4", "FAH", "GDF15",
    "SQSTM1", "ALDOB", "GSTT1", "MAP2K2", "BCAR1", "SP1", "SLC67A1", "NME1", "THADA", "LDHA",
    "FGF2", "HMGB1", "UFC1", "EPHB2", "SLC11A2", "NCOR1", "MMP14", "IRS2", "CLDN1", "IL1A",
    "CYCS", "FOLH1", "GFER", "TGFA", "CCN2", "WEE1", "FOS", "NOTCH3", "ABCC3", "DCC",
    "STAT5B", "RPS20", "TET2", "KLK3", "KMT2A", "PSCA", "IL7R", "TGFB2", "ESR2", "IL6R",
    "SOS1", "ZEB1", "EPHB4", "BIVM-ERCC5", "UBAP2", "PKM", "CASP10", "DNMT3B", "SLC2A2", "CYP3A5",
    "BMP2", "CTSD", "PLAUR", "HSPB1", "CD36", "CCND3", "KLLN", "OTC", "ATG7", "PTPRC",
    "ITGA5", "ATP8B1", "CCND2", "SOX4", "PRKCD", "BECN1", "LIG4", "OGG1", "CDH17", "BMI1",
    "FLNA", "ENO1", "ZNF281", "BMP4", "NOS2", "ETS1", "CCNB1", "TIMP2", "PPP2R2A", "IFNA1",
    "BMP7", "EXT2", "AREG", "PPARGC1A", "HELZ", "FBP1", "PPP2R1B", "TF", "PRF1", "LGALS3",
    "AMACR", "RUNX3", "EGR1", "BAK1", "ZFR", "CSF1R", "PAX8", "BCL10", "RUVBL1", "APEX1",
    "MAPK14", "CYP2B6", "CYP2C9", "FOXA2", "IKBKB", "RB1CC1", "YBX1", "BID", "NR1I2", "WNT5A",
    "CD40LG", "HIPK3", "MMUT", "NCOA3", "FGF19", "PIK3CD", "SLC19A1", "ELAVL1", "TMEM238L", "UGT1A9",
    "CEBPB", "KIF1B", "NR3C1", "HLA-B", "SNAI2", "MYD88", "AURKB", "PTPN3", "PFKL", "ATG5",
    "WRAP53", "CDH2", "PKLR", "PDGFB", "POU5F1", "ACACA", "DKK1", "CCL5", "KDM1A", "TNFSF11",
    "PC", "HLA-A", "NR5A2", "ICAM1", "EPAS1", "PSTPIP1", "HAVCR2", "NEK9", "BCL2L11", "EPHX1",
    "LIPA", "MXI1", "BRD4", "NSD1", "ADH1B", "CYP7A1", "RIPK1", "TSG101", "SLC37A4", "PLAT",
    "MTR", "MUC6", "STAT6", "PRKACA", "TOE1", "DCTN5", "POU3F3", "ASS1", "GUSB", "CDX2",
    "CEP290", "MBD4", "BTK", "ALDOA", "CPS1"
]

print(f"Number of input genes: {len(liver_cancer_genes)}")
print(f"First 10 genes: {liver_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(liver_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Liver Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('liver_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: liver_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('liver_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: liver_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'liver_cancer_core_network.graphml')
print("✓ Core network saved: liver_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Liver Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. liver_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. liver_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. liver_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Lung Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Lung Cancer Gene List ====================
print("="*70)
print("Lung Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Lung cancer-associated gene list
lung_cancer_genes = [
    "BRCA2", "EGFR", "BRCA1", "ATM", "TP53", "PALB2", "CHEK2", "MLH1", "BRIP1", "APC",
    "MSH2", "MSH6", "PTEN", "CDH1", "ALK", "ERBB2", "BARD1", "DICER1", "BRAF", "TERT",
    "TSC2", "NF1", "CDKN2A", "C11orf65", "KRAS", "PMS2", "STK11", "POLD1", "RB1", "POLE",
    "MET", "RET", "PIK3CA", "SMARCA4", "AXIN2", "SMAD4", "TSC1", "AKT1", "RAD50", "CTNNB1",
    "BAP1", "KIT", "NBN", "PDGFRA", "MUTYH", "PTCH1", "RAD51D", "RAD51C", "CCND1", "MSH3",
    "MYC", "BLM", "CD274", "HRAS", "CDK4", "FLCN", "ESR1", "FGFR3", "CTNNA1", "STAT3",
    "EPCAM", "FGFR2", "BMPR1A", "MEN1", "MRE11", "CDKN1B", "NKX2-1", "MTOR", "ATR", "MDM2",
    "FGFR1", "RUNX1", "NRAS", "VHL", "CASP8", "SDHA", "FANCC", "IL6", "NOTCH1", "POT1",
    "HIF1A", "NFE2L2", "ROS1", "FH", "TGFB1", "LZTR1", "VEGFA", "HOXB13", "MAP2K1", "ERBB3",
    "TGFBR2", "SDHB", "ERCC2", "CHEK1", "PDCD1", "FASLG", "ERBB4", "MLH3", "MUC16", "BCL2",
    "NTHL1", "AR", "NF2", "ARID1A", "BAX", "SRC", "MUC1", "SMARCB1", "FBXW7", "RASSF1",
    "PPARG", "FAS", "RAD51", "NTRK1", "TUG1", "XRCC2", "ERCC6", "PRKN", "ABCA3", "PTGS2",
    "IL1B", "IGF1R", "EZH2", "KEAP1", "ERCC1", "SOX2", "CXCR4", "CDC73", "PRKAR1A", "PIK3CG",
    "SUFU", "TNF", "RAF1", "NRG1", "SDHD", "KMT2D", "CFTR", "CASP3", "BIRC5", "CDKN2B",
    "SMAD3", "KDR", "EP300", "IDH1", "GREM1", "CDKN1A", "IRF1", "DLEC1", "FOXO1", "CXCL8",
    "SDHC", "WT1", "RARB", "SOD2", "MMP9", "SFTPC", "MMP2", "AOPEP", "MAPK1", "ACTA2",
    "GATA3", "BUB1B", "CD44", "CTLA4", "ITGA3", "ABCB1", "PIK3R1", "AXL", "NTRK3", "CDK6",
    "CCNE1", "PTK2", "MAX", "CDK12", "TMEM127", "GNAS", "XRCC1", "IDH2", "IFNG", "GSTP1",
    "SMAD2", "FANCA", "JUN", "DLC1", "FANCD2", "NTRK2", "FANCM", "ERCC5", "RECQL4", "TP63",
    "XPC", "RNF43", "FLT1", "STAT1", "TNFSF10", "GSTM1", "CAV1", "MITF", "CYP1A1", "TP73",
    "MGMT", "IL10", "DDR2", "AURKA", "SLC67A1", "DHFR", "PMS1", "JAK2", "CYP2A6", "SFTPA1",
    "SMO", "SDHAF2", "ABCC1", "ANXA2", "CCL2", "GLI1", "LRP1B", "NFKB1", "FHIT", "TLR4",
    "FGF10", "ABCG2", "PHOX2B", "WRN", "ERCC4", "MUC5B", "PPP2R1B", "XRCC3", "PARP1", "ENG",
    "SMARCE1", "SOX9", "COPA", "MARS1", "GATA2", "DNMT1", "SFTPB", "IL2", "PPM1D", "AKT3",
    "IL1RN", "TGFBR1", "PTPN11", "RAD51B", "SLC19A1", "IGF2", "BCL2L1", "MMP1", "SNAI1",
    "XPA", "FOXO3", "EGF", "IGF1", "YAP1", "METTL3", "PRKDC", "NFKBIA", "HMOX1", "SHH",
    "GSK3B", "FLNA", "DNMT3A", "CDKN1C", "SPOP", "ITGB1", "MTHFR", "SMAD7", "AKT2", "SERPINA1",
    "CREBBP", "HGF", "TYMS", "E2F1", "MCL1", "FGFR4", "FOXA1", "AIP", "RHOA", "RTEL1",
    "GPC3", "RELA", "CTAG1B", "CASP9", "GALNT12", "ERCC3", "NSD2", "CXCL12", "MKI67", "NQO1",
    "KLF6", "KRT7", "AGO2", "VEGFC", "FN1", "FLT4", "RAD54L", "ENO1", "CDK2", "CYP17A1",
    "PDGFRB", "SFTPA2", "XIAP", "TMPRSS2", "SHC1", "TLR2", "TWIST1", "NOTCH3", "FOXP3", "GSTM3",
    "TCF7L2", "KRT20", "FANCG", "PPP2R1A", "SLC2A1", "SETD2", "MXRA5", "ZEB1", "CBL", "CEBPA",
    "BIRC3", "RNASEL", "MUC5AC", "IL4", "ATRX", "MAP3K8", "SPP1", "CD82", "BLACAT1", "FANCE",
    "DDB2", "TOP2A", "FOXF1", "PROM1", "AXIN1", "WRAP53", "CCN2", "PKHD1", "FANCL", "MTA1",
    "BUB1", "SERPINE1", "CDK1", "MAP2K2", "PALLD", "RECQL", "TFF1", "U2AF1", "CLPTM1L", "NPM1",
    "CEACAM5", "ESR2", "IGFBP3", "SOX4", "ZNF609", "FGF2", "MAP3K1", "TNFRSF10B", "ZEB2", "FANCF",
    "TIMP1", "HSPA5", "GSTT1", "STING1", "PLAU", "MMP7", "CYP19A1", "HDAC1", "CSF3", "WWOX",
    "CYP1B1", "MYCN", "GNAQ", "MAPK3", "LRRC56", "GNA11", "ELAC2", "TOP1", "PRKCA", "BMP4",
    "TINCR", "OGG1", "APEX1", "BMP2", "CASP10", "DSP", "MSR1", "FGF7", "WEE1", "CCR2",
    "SIRT1", "HNF1B", "THADA", "MAP2K4", "MT-CYB", "FANCI", "ERG", "HMGB1", "VDR", "CREB1",
    "KLF4", "ITCH", "FOXM1", "RUBCNL", "MMP14", "PCNA", "ADAM12", "PGR", "NME1", "HSP90AA1",
    "NOTCH2", "PDPN", "SLC34A2", "HNF1A", "ABRAXAS1", "KRT19", "PGBD3", "DKC1", "ALDH1A1", "MPO",
    "PPP2R2A", "CSF2", "POLK", "RARA", "MAPK8", "PLAUR", "AHR", "VIM", "BCL10", "NCAPG",
    "SP1", "ICAM1", "SOCS1", "HLA-DRB1", "SLTM", "TGFB2", "BMP6", "IL7R", "HSPB1", "LIG4",
    "SFTPD", "JAK1", "CHRNA5", "RHBDF2", "SCGB1A1", "HMGA2", "RUNX1T1", "ADIPOQ", "CCNB1", "IL2RA",
    "EPHA5", "TGFA", "ELANE", "CCND3", "NAT2", "LMNA", "S100B", "PIK3CD", "AREG", "CDH2",
    "CBS", "ENO2", "CCL5", "IL1A", "ITGA5", "NCOA3", "CADM1", "SMARCA5", "STAT6", "MAPK14",
    "RRM1", "CYP24A1", "HMMR", "DNMT3B", "PDGFB", "ETS1", "NCOR1", "COL3A1", "RUNX3", "BRD4",
    "EML4", "SNAI2", "DPYD", "SOS1", "KDM6A", "PTHLH", "BMPR2", "SEPTIN9", "GRP", "BAK1",
    "BCL2L11", "SLX4", "CCND2", "BCAR1", "MYCL", "EPHB2", "HLA-B", "TINF2", "ABCC4", "ABL1",
    "CDC25B", "EPHB4", "BMI1", "AURKB", "EPHX1", "POSTN", "DROSHA", "CASR", "H2AX", "MBD4",
    "IL13", "NCAPG2", "WNT5A", "TIMP2", "ACD", "PHB1", "PBRM1", "JAK3", "KLK3", "DCC",
    "CDKN3", "NOS2", "SPARC", "ELAVL1", "CSF1R", "KMT2A", "PSCA", "CTCF", "LAMC2", "ALDH2",
    "LAMA5", "IKBKB", "CD24", "GDF15", "PIK3CB", "MXI1", "PRMT5", "TUBB3", "ADAM17", "KITLG",
    "BCL6", "CYCS", "ELN", "CFLAR", "ABCC2", "KAT7", "NLRP3", "IL17A", "TPX2", "PDCD1LG2",
    "GRM8", "IL33", "BECN1", "HLA-A", "CA9", "FBN1", "ITGA2", "FADD", "DDR1", "PAX5",
    "RAC1", "CHGA", "CYP2E1", "MVP", "CKS1B", "ADAR", "CSF1", "HFE", "S100A4", "ASCL1",
    "PLK1", "CYP3A4", "MYD88", "NSD1", "TET2", "CRP", "EGR1", "KLF2", "MTLN", "DKK1",
    "MYLK", "ID1", "TIMP3", "EDN1", "CHUK", "TLR9", "TYMP", "MAGEA3", "MSLN", "UFC1",
    "SPINK1", "FASN", "KDM1A", "CD40", "LEP", "FOLH1", "ETV4", "KDM4B", "MUC4", "SOCS3",
    "YBX1", "ODC1", "CTSD", "PKM", "EXO1", "IGF2R", "FLT3", "CDC25A", "EPAS1", "PTPRC",
    "PTK2B", "CDX2", "IRS1", "ALB", "COL1A1", "EPHA2", "LOX", "DAPK1", "MIF", "NEK9",
    "BSG", "CHRNA3", "PARN", "DCTN5", "THBS1", "IL18", "KRT5", "FOS", "BAD", "NCAM1",
    "CD40LG", "TNFRSF10A", "SCUBE3", "XRCC6", "NT5E", "CEACAM3", "ADGRB3", "MUC6", "VEGFD", "LGALS3",
    "MCC", "KRT18", "SOD1", "CXCL1", "CCR7", "MDM4", "RPS20", "KMT2C", "GATA6", "GJA1",
    "PTPRG", "DDIT3", "BID", "ABCA1", "ACE", "CCNA2", "CP", "EXT1", "TOE1", "YWHAE",
    "BRMS1", "GLI2", "GADD45A", "UBR1", "BIVM-ERCC5", "CTC1", "MMP3", "CEACAM6", "FAM13A", "TNFRSF1A",
    "B2M", "HDAC9", "EXT2", "KLLN", "CD8A", "CXCL10", "EWSR1", "SMARCA2", "JAG1", "ACACA",
    "RIPK1", "CHST15", "GRB2", "NRP1", "YY1", "HAVCR2", "TRIM24", "IGF2BP3", "STAT5B", "IL6R",
    "PLAT", "PAK4", "EPHA3", "IQANK1", "RPS6KB1", "MAD1L1", "PIK3R2", "CDC25C", "CD36", "CALCA",
    "KRT8", "COMT", "HSPA4", "RB1CC1", "MBL2", "ZFHX3", "SLMAP", "POU5F1", "AGER", "CXCR3",
    "BTK", "ING1", "TSHR", "E2F3", "CDC42", "DIABLO", "PLA2G2A", "BIRC7", "ATG5", "TSG101",
    "SLC26A4", "RPA1", "ROBO1", "ARAF", "ACVR1B", "ALOX5", "PRKACA", "XRCC5", "ETV6", "FSCN1",
    "BIRC2", "CTAG2", "ANGPT1", "STAT5A", "MTR", "HIPK3", "CYP1A2", "CXCR2", "HDAC6", "ANGPT2",
    "SKP2"
]

print(f"Number of input genes: {len(lung_cancer_genes)}")
print(f"First 10 genes: {lung_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(lung_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Lung Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('lung_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: lung_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('lung_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: lung_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'lung_cancer_core_network.graphml')
print("✓ Core network saved: lung_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Lung Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. lung_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. lung_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. lung_cancer_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Mesothelioma Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Mesothelioma Gene List ====================
print("="*70)
print("Mesothelioma Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Mesothelioma-associated gene list
mesothelioma_genes = [
    "WT1", "BAP1", "BCL10", "CALB2", "MSLN", "CDKN2A", "NF2", "LATS2", "LATS1", "THBD",
    "VIM", "MUC1", "PDPN", "NKX2-1", "CD274", "CTNNB1", "KRT5", "TP53", "BCL2L1", "MET",
    "SPP1", "CEACAM5", "CEACAM3", "MTAP", "KRT7", "CDH1", "EGFR", "TNFSF10", "SFRP4", "CDH2",
    "FGF2", "DES", "HGF", "RASSF1", "TYMS", "SOD2", "MUC16", "CDK4", "KDR", "BRCA2",
    "MCL1", "PDGFRB", "FLT1", "DNAH1", "KRT20", "SLC19A1", "TNFRSF10B", "FLT4", "AREG", "TXN",
    "PDGFA", "TP63", "CDKN2B", "IFNB1", "KRT19", "MTOR", "BIRC5", "CD34", "DHFR", "GPC3",
    "FOLR1", "SNAI2", "CLDN4", "HAS3", "PECAM1", "PAX8", "CD7", "AKT1", "ALK", "VEGFA",
    "YAP1", "EFEMP1", "BCL2", "EZH2", "TNFRSF8", "RNASE1", "PTEN", "SETD2", "RNF43", "PHF7",
    "MMP9", "PDCD1", "TRAF7", "SLC2A1", "HEG1", "CD44", "MDM2", "XIAP", "HMGB1", "BRCA1",
    "HIF1A", "GATA3", "TGFB1", "ERCC1", "GSTM1", "KIT", "ATM", "MAPK1", "BAX", "FGFR1",
    "IGF2", "DPP4", "EPCAM", "SLTM", "PTK2", "CASP3", "CCND1", "IL6", "FOXM1", "EIF4EBP1",
    "STRN", "CTLA4", "GLI1", "IGF2BP3", "SPARC", "CXCL8", "IL1B", "PTGS2", "CEACAM7", "MYC",
    "TERT", "SETDB1", "CA9", "PPP1R15A", "UHRF1", "MMP14", "NAPSA", "CCN2", "CDKN1A", "PIK3CA",
    "WWTR1", "DDIT3", "RRM1", "PTGER4", "PARP1", "VHL", "CTNNA1", "E2F1", "PGF", "FGFR2",
    "WIF1", "FGF1", "YY1", "SDC1", "MMP2", "MAPK7", "BAK1", "MUC4", "STAT3", "CREB1",
    "EGF", "MAPK3", "CSF1R", "ASS1", "SRC", "MCAM", "SFRP2", "CCL2", "STAT6", "IL34",
    "TGFA", "ITLN1", "SFRP1", "TSC1", "FN1", "EIF2AK3", "CSF3", "PDGFD", "LSM1", "GCLM",
    "XRCC1", "STAT1", "COL1A2", "ABCB1", "AXL", "BRAF", "FOSL1", "ATG13", "CASP8", "BID",
    "CXCL12", "EWSR1", "SGPP1", "VEGFC", "DKK1", "ABCG2", "PLAUR", "STMN1", "TEAD1", "NAB2",
    "TLE1", "MKI67", "BARD1", "ANGPT1", "NAT2", "SP1", "FPGS", "KITLG", "GSTT1", "MAD2L1",
    "CD86", "HRAS", "CCND2", "MAP2K5", "IL2", "CLDN15", "FOXO1", "CXCL10", "MAPK8", "ACTA2",
    "FOXP3", "DVL3", "DDX51", "LIAT1", "HSP90AA1", "APC", "ERBB2", "CAT", "WNT7A", "PLAU",
    "IFNA2", "PMAIP1", "BMP6", "RELA", "ASXL1", "KDM6A", "MMP3", "PAX2", "GSTP1", "DNMT1",
    "NLRP3", "CSNK2A1", "CSNK2A2", "CD99"
]

print(f"Number of input genes: {len(mesothelioma_genes)}")
print(f"First 10 genes: {mesothelioma_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(mesothelioma_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Mesothelioma (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('mesothelioma_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: mesothelioma_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('mesothelioma_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: mesothelioma_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'mesothelioma_core_network.graphml')
print("✓ Core network saved: mesothelioma_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Mesothelioma Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. mesothelioma_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. mesothelioma_all_genes_metrics.csv - Network metrics for all genes")
print("  3. mesothelioma_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Nasopharyngeal Carcinoma Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Nasopharyngeal Carcinoma Gene List ====================
print("="*70)
print("Nasopharyngeal Carcinoma Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Nasopharyngeal carcinoma-associated gene list
nasopharyngeal_carcinoma_genes = [
    "TP53", "ATM", "BRCA1", "MET", "PALB2", "CHEK2", "BRCA2", "BRIP1", "MSH2", "EGFR",
    "MLH1", "BARD1", "CTNNB1", "PMS2", "CDH1", "PTEN", "C11orf65", "BRAF", "CDKN2A", "ERBB2",
    "RET", "PIK3CA", "APC", "KRAS", "CCND1", "AKT1", "VHL", "TERT", "MTOR", "SMAD4",
    "PTCH1", "HRAS", "CDC73", "STK11", "TGFBR2", "FGFR3", "RB1", "MYC", "CD274", "HIF1A",
    "HNF1A", "ALK", "VEGFA", "STAT3", "EPCAM", "TUG1", "ESR1", "KIT", "MDM2", "SMARCA4",
    "CDK4", "MSH3", "BAX", "NOTCH1", "NRAS", "ARID1A", "FGFR2", "TSC2", "CASP8", "KRT7",
    "CDKN1B", "BIRC5", "PTGS2", "IL6", "BCL2", "MUTYH", "FGFR1", "NKX2-1", "TP63", "POLE",
    "MST1R", "TGFB1", "FH", "PBRM1", "TSC1", "CTNNA1", "CDKN1A", "NTRK1", "CASP3", "NFE2L2",
    "RASSF1", "MMP9", "CD44", "SDHB", "KDR", "KRT20", "ITCH", "NBN", "EP300", "NFKBIA",
    "RAD50", "DLC1", "SMARCB1", "SRC", "FASLG", "FBXW7", "PDCD1", "BAP1", "CDKN2B", "MUC1",
    "NF1", "TINCR", "AR", "SMO", "MEN1", "PDGFRA", "DICER1", "XRCC1", "NUTM1", "ZNF609",
    "OGG1", "IGF1R", "ERCC2", "SETD2", "MMP2", "SOX2", "RAD51", "RAD51C", "CA9", "JUN",
    "EZH2", "SDHD", "CXCL8", "ERBB3", "FAS", "CHEK1", "CXCR4", "BUB1B", "POU3F3", "PPARG",
    "TNFRSF10B", "ABCC1", "EGF", "SDHC", "BCL2L1", "IL2", "FHIT", "MAP2K1", "STAT1", "PCNA",
    "WWOX", "MKI67", "IFNG", "NTRK3", "VIM", "ERCC1", "TWIST1", "TNF", "CDK6"
]

print(f"Number of input genes: {len(nasopharyngeal_carcinoma_genes)}")
print(f"First 10 genes: {nasopharyngeal_carcinoma_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(nasopharyngeal_carcinoma_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Nasopharyngeal Carcinoma (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('nasopharyngeal_carcinoma_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: nasopharyngeal_carcinoma_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('nasopharyngeal_carcinoma_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: nasopharyngeal_carcinoma_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'nasopharyngeal_carcinoma_core_network.graphml')
print("✓ Core network saved: nasopharyngeal_carcinoma_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Nasopharyngeal Carcinoma Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. nasopharyngeal_carcinoma_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. nasopharyngeal_carcinoma_all_genes_metrics.csv - Network metrics for all genes")
print("  3. nasopharyngeal_carcinoma_core_network.graphml - Cytoscape network file")
print("="*70)

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Skin Cancer Core Hub Gene PPI Network Analysis
Screening of True Top 10 Hub Genes via Multi-Criteria Comprehensive Scoring
(Pure Screening Version, No Plotting)
"""

import requests
import networkx as nx
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

# ==================== 1. Input Skin Cancer Gene List ====================
print("="*70)
print("Skin Cancer Core Hub Gene Screening and Analysis (Pure Screening Version)")
print("="*70)

# Skin cancer-associated gene list
skin_cancer_genes = [
    "BRCA2", "BRCA1", "ATM", "PALB2", "TP53", "CHEK2", "BRIP1", "EGFR", "MSH2", "MSH6",
    "MLH1", "CDH1", "APC", "BARD1", "NF1", "PTEN", "PMS2", "C11orf65", "BRAF", "POLD1",
    "AXIN2", "CDKN2A", "POLE", "DICER1", "ERBB2", "TSC2", "ALK", "RAD50", "PTCH1", "RET",
    "STK11", "MET", "PIK3CA", "SMAD4", "KIT", "SMARCA4", "RB1", "NBN", "KRAS", "MUTYH",
    "RAD51C", "BLM", "BAP1", "RAD51D", "CDK4", "MSH3", "CTNNB1", "AKT1", "TSC1", "HRAS",
    "PDGFRA", "FLCN", "CTNNA1", "TERT", "EPCAM", "FBN1", "FGFR3", "BMPR1A", "MEN1", "MRE11",
    "RUNX1", "NRAS", "FH", "FGFR2", "SDHA", "CDKN1B", "FANCC", "ESR1", "CCND1", "LZTR1",
    "MYC", "VHL", "TYR", "MTOR", "HOXB13", "STAT3", "SDHB", "ERCC2", "IL6", "AR",
    "ATR", "NF2", "DSP", "TGFBR2", "POT1", "NTHL1", "CD274", "COL7A1", "SMARCB1", "MITF",
    "MLH3", "PRKAR1A", "MDM2", "SUFU", "HIF1A", "NOTCH1", "PDGFRB", "SDHD", "XRCC2", "MC1R",
    "RECQL4", "TGFB1", "VEGFA", "CDC73", "SDHC", "FGFR1", "MAP2K1", "RAD51", "PDCD1", "TP63",
    "ADAM17", "CASP8", "BCL2", "CTLA4", "IL1B", "ARID1A", "TMEM127", "GREM1", "TNF", "OCA2",
    "FANCM", "CHEK1", "ERCC5", "SRC", "BAX", "EZH2", "SMAD3", "LMNA", "PIK3R1", "RAG1",
    "KRT14", "AOPEP", "IDS", "XPC", "FANCD2", "KDR", "IFNG", "ERCC4", "NFE2L2", "WRN",
    "EP300", "WT1", "FANCA", "ERBB3", "SOX2", "FASLG", "SDHAF2", "ERBB4", "NTRK1", "PKP1",
    "CXCR4", "PTGS2", "PPARG", "MAX", "XPA", "SMAD2", "STAT1", "RAF1", "PMS1", "PTPN11",
    "MUC16", "IL10", "MMP2", "MAPK1", "PHOX2B", "BUB1B", "TUG1", "TGFBR1", "CDKN1A", "CDKN2B",
    "ERCC3", "SMARCE1", "COL1A1", "SMO", "MMP9", "MMP1", "RAG2", "AIP", "FAS", "MUC1",
    "JUN", "ERCC1", "CASP3", "CD44", "DLC1", "FBXW7", "IL2", "TGM5", "GNAS", "GATA2",
    "IDH1", "CCNE1", "IGF1R", "KITLG", "DHFR", "RNF43", "JAK2", "KRT5", "CDK6", "BIRC5",
    "SHH", "IL1RN", "GATA3", "NFKBIA", "ITGB4", "GLI1", "DDB2", "SLC45A2", "PPM1D", "XRCC3",
    "COL3A1", "CXCL8", "ABCA12", "ENG", "RAD51B", "FANCG", "FOXP3", "FLG", "PIK3CG", "IGF1",
    "IDH2", "AURKA", "EGF", "ABCB1", "MGMT", "NFKB1", "TLR4", "CDK12", "GALNT12", "FANCL",
    "DNMT3A", "LAMC2", "KMT2D", "ERCC6", "TYRP1", "RARB", "TNFSF10", "GPC3", "SOD2", "AKT3",
    "IGF2", "PTK2", "FOXO1", "GJA1", "ACTA2", "CCL2", "FANCE", "XRCC1", "MAP2K2", "CREBBP",
    "KRT10", "RECQL", "DNMT1", "FANCI", "CDSN", "SPOP", "SOX9", "CDKN1C", "ROS1", "FLT4",
    "PARP1", "KRT1", "RAD54L", "PRKDC", "NRG1", "TGFB2", "FANCF", "KRT20", "ITGB1", "NTRK3",
    "HNF1A", "IRF1", "CEBPA", "ELN", "ATRX", "MTHFR", "GSTP1", "PLEC", "RHBDF2", "SMAD7",
    "TLR2", "YAP1", "CBL", "VEGFC", "KRT7", "KLF4", "ABCC1", "LOX", "GNA11", "GNAQ",
    "ABCG2", "PALLD", "KRT17", "COL1A2", "SLX4", "COL17A1", "LAMB3", "FLT1", "MKI67", "ABCC6",
    "FN1", "VDR", "GSTM1", "IL4", "FERMT1", "HNF1B", "RASSF1", "TGM1", "RNASEL", "HLA-DRB1",
    "CSTA", "NTRK2", "JAK1", "ATP6V0A2", "NPM1", "TWIST1", "AKT2", "ZEB2", "GJB2", "XIAP",
    "FGFR4", "BMP4", "FLNA", "FOXO3", "TYMS", "NOTCH3", "MT-CYB", "CFTR", "GSK3B", "CYP17A1",
    "PRKN", "FBLN5", "MAP3K1", "CBS", "CYLD", "FHIT", "NKX2-1", "TFF1", "IL1A", "SHC1",
    "CTAG1B", "CXCL12", "ERG", "PDGFB", "BCL2L1", "DSG1", "RHOA", "ITGA6", "SOS1", "HLA-B",
    "ITCH", "BIVM-ERCC5", "LRRC56", "CASR", "FGF10", "ABRAXAS1", "KEAP1", "SLC2A1", "TUBB", "CDH3",
    "HGF", "AGO2", "CCN2", "CAV1", "FGF2", "JUP", "AXL", "FLG2", "ZMPSTE24", "IL2RA",
    "ADAR", "FOXA1", "COL5A1", "WWOX", "PCNA", "CYP19A1", "CYP1A1", "RELA", "SPP1", "TCF7L2",
    "KLF6", "KDM6A", "KLLN", "E2F1", "BUB1", "SNAI2", "SNAI1", "MCL1", "AXIN1", "ABL1",
    "ADIPOQ", "CDK2", "NOTCH2", "SEPTIN9", "MMP14", "IGFBP3", "SETD2", "WEE1", "KMT2A", "BLACAT1",
    "SOX10", "BMP6", "ELAC2", "PIK3CD", "ETS1", "HMMR", "PGR", "LRP1B", "DCC", "RARA",
    "IL7R", "PRKCA", "TGFB3", "DKC1", "HDAC1", "CASP10", "TRPS1", "ODC1", "EPHB4", "SOCS1",
    "CTSB", "HSPB1", "NSD1", "PROM1", "EXT2", "ELANE", "TWIST2", "IL17A", "MYLK", "BCL10",
    "PTCH2", "MYD88", "TMPRSS2", "MSR1", "WRAP53", "CASP9", "PYCR1", "TNFRSF10B", "MAPK3", "PRF1",
    "STAT6", "HFE", "IRF4", "ESR2", "AHR", "TET2", "ST14", "JAK3", "CDK1", "TP73",
    "CD40LG", "TNFRSF1A", "EPHB2", "HLA-A", "PLAU", "IAH1", "BMP2", "NQO1", "ABCA1", "VIM",
    "EXT1", "SERPINE1", "TINCR", "MAP2K4", "CD36", "DST", "KRT19", "TIMP1", "PBRM1", "WNT5A",
    "ANXA2", "HSP90AA1", "PHB1", "DLEC1", "ETV6", "HMGA2", "GLI2", "KLK3", "NCOR1", "AREG",
    "IL33", "S100B", "NOD2", "LDLR", "MTA1", "NME1", "HCCS", "ICAM1", "PTHLH", "CREB1",
    "CCND2", "FGF7", "PNPLA1", "MME", "SPINK5", "RPS20", "ACTB", "INSR", "SMARCA2", "SLC19A1",
    "METTL3", "CD28", "MYCN", "LIG4", "KRT2", "PDPN", "FOLH1", "HERC2", "POLH", "CLDN1",
    "GSTM3", "BIRC3", "PSCA", "POMC", "KMT2C", "FOS", "HSPA5", "PTPRC", "TEK", "MYH11",
    "SIRT1", "JAG1", "MAPRE2", "TBX3", "SPARC", "DDR2", "SHOC2", "CCL5", "MMP7", "BMI1",
    "TNFAIP3", "NLRP3", "CYP24A1", "CD40", "ZEB1", "RASA1", "BCAR1", "INS", "TOP2A", "CD82",
    "IL18", "DCTN5", "CYP1B1", "ASPRV1", "EFEMP2", "BTK", "DSC3", "CSF1R", "LEF1", "CSF3",
    "RINT1", "PORCN", "KDM1A", "CD4", "CSF2", "TOE1", "MED12", "PAX5", "ATP7A", "TGFA",
    "DIS3L2", "LORICRIN", "IL13", "MAPK14", "SASH1", "RTEL1", "STAT5B", "CHUK", "TINF2", "ALB",
    "BCOR", "KIF1B", "ALDH2", "HMOX1", "FLT3", "CAST", "MMP13", "GSTT1", "CEACAM5", "MAPK8",
    "IKBKB", "APEX1", "ITGA5", "HMGB1", "SP1", "KRT16", "RRAS2", "SPRED1", "DPYD", "RB1CC1",
    "IQANK1", "CD8A", "MMP3", "TOP1", "HLA-DQB1", "IL7", "DNMT3B", "LEP", "TSHR", "OGG1",
    "ATRIP", "RIPK1", "ITGA2", "SYK", "PLAUR", "CCND3", "H2AX", "STING1", "NEK9", "B2M",
    "CRP", "FADD", "LAMA3", "CP", "CDKN3", "IKBKG", "MAPK10", "FANCB", "EWSR1", "BRD4",
    "IFNA1", "POSTN", "NSD2", "ABCC4", "CXCL10", "CDC25B", "RBBP8", "PPP2R2A", "CDH2", "NT5E",
    "TIMP2", "PIK3CB"
]

print(f"Number of input genes: {len(skin_cancer_genes)}")
print(f"First 10 genes: {skin_cancer_genes[:10]}")

# ==================== 2. Retrieve PPI Data from STRING Database ====================
print("\n" + "="*70)
print("Phase 1: Retrieving PPI Data from STRING Database")
print("="*70)

def get_string_ppi(genes, species="9606", score_threshold=0.7):
    """Retrieve PPI data from STRING database"""
    string_api_url = "https://string-db.org/api"
    batch_size = 100
    all_interactions = []
    
    for i in range(0, len(genes), batch_size):
        batch = genes[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(genes)-1)//batch_size + 1}...")
        
        params = {
            'identifiers': '%0d'.join(batch),
            'species': species,
            'limit': 20,
            'required_score': int(score_threshold * 1000),
        }
        
        try:
            response = requests.post(f"{string_api_url}/json/network", data=params)
            if response.status_code == 200:
                data = response.json()
                all_interactions.extend(data)
                print(f"  ✓ Retrieved {len(data)} interactions")
            else:
                print(f"  ✗ Request failed: {response.status_code}")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        
        time.sleep(1)
    
    return all_interactions

def build_network_from_string(interactions):
    """Construct NetworkX graph from STRING data"""
    G = nx.Graph()
    
    for item in interactions:
        if 'preferredName_A' in item and 'preferredName_B' in item:
            protein_a = item['preferredName_A']
            protein_b = item['preferredName_B']
            score = float(item.get('score', 0))
            
            G.add_node(protein_a)
            G.add_node(protein_b)
            G.add_edge(protein_a, protein_b, score=score)
    
    return G

# Retrieve PPI data
interactions = get_string_ppi(skin_cancer_genes, score_threshold=0.7)
print(f"\nTotal interactions retrieved: {len(interactions)}")

# Construct network
G = build_network_from_string(interactions)

print(f"\n✓ Network construction completed!")
print(f"  Number of nodes (proteins): {G.number_of_nodes()}")
print(f"  Number of edges (interactions): {G.number_of_edges()}")
print(f"  Network density: {nx.density(G):.4f}")

# ==================== 3. Calculate Network Topological Metrics ====================
print("\n" + "="*70)
print("Phase 2: Calculating Network Topological Metrics")
print("="*70)

# Degree
degrees = dict(G.degree())
print("✓ Degree calculation completed")

# Degree centrality
degree_centrality = nx.degree_centrality(G)
print("✓ Degree centrality calculation completed")

# Betweenness centrality
print("Calculating betweenness centrality...")
k = min(100, G.number_of_nodes())
betweenness_centrality = nx.betweenness_centrality(G, k=k)
print(f"✓ Betweenness centrality calculation completed (k={k})")

# Closeness centrality
closeness_centrality = nx.closeness_centrality(G)
print("✓ Closeness centrality calculation completed")

# Eigenvector centrality
try:
    eigenvector_centrality = nx.eigenvector_centrality_numpy(G)
    print("✓ Eigenvector centrality calculation completed (numpy method)")
except:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=1000)
    print("✓ Eigenvector centrality calculation completed (iterative method)")

# PageRank
pagerank = nx.pagerank(G)
print("✓ PageRank calculation completed")

# ==================== 4. Multi-Criteria Comprehensive Scoring for Hub Gene Screening ====================
print("\n" + "="*70)
print("Phase 3: Multi-Criteria Comprehensive Scoring for Hub Gene Screening")
print("="*70)

# Create metrics DataFrame
df_metrics = pd.DataFrame({
    'Gene': list(degrees.keys()),
    'Degree': list(degrees.values()),
    'Degree_Centrality': [degree_centrality[g] for g in degrees.keys()],
    'Betweenness_Centrality': [betweenness_centrality[g] for g in degrees.keys()],
    'Closeness_Centrality': [closeness_centrality[g] for g in degrees.keys()],
    'Eigenvector_Centrality': [eigenvector_centrality[g] for g in degrees.keys()],
    'PageRank': [pagerank[g] for g in degrees.keys()]
})

print(f"Total number of genes for scoring: {len(df_metrics)}")

# Calculate Z-scores for each metric
print("\nCalculating Z-scores for each metric...")
scaler = StandardScaler()
metrics_for_scoring = ['Degree', 'Betweenness_Centrality', 
                       'Eigenvector_Centrality', 'PageRank']

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_metrics[metrics_for_scoring]),
    columns=[f'{col}_zscore' for col in metrics_for_scoring],
    index=df_metrics.index
)

# Calculate composite score with weights
weights = {
    'Degree_zscore': 0.35,
    'Betweenness_Centrality_zscore': 0.25,
    'Eigenvector_Centrality_zscore': 0.20,
    'PageRank_zscore': 0.20
}

df_metrics['Zscore_Degree'] = df_scaled['Degree_zscore']
df_metrics['Zscore_Betweenness'] = df_scaled['Betweenness_Centrality_zscore']
df_metrics['Zscore_Eigenvector'] = df_scaled['Eigenvector_Centrality_zscore']
df_metrics['Zscore_PageRank'] = df_scaled['PageRank_zscore']

# Calculate weighted composite score
df_metrics['Composite_Score'] = (
    df_scaled['Degree_zscore'] * weights['Degree_zscore'] +
    df_scaled['Betweenness_Centrality_zscore'] * weights['Betweenness_Centrality_zscore'] +
    df_scaled['Eigenvector_Centrality_zscore'] * weights['Eigenvector_Centrality_zscore'] +
    df_scaled['PageRank_zscore'] * weights['PageRank_zscore']
)

# Sort by composite score
df_metrics_sorted = df_metrics.sort_values('Composite_Score', ascending=False).reset_index(drop=True)

# Display Top 20 candidate genes
print("\n📊 Top 20 Candidate Hub Genes for Skin Cancer (Ranked by Composite Score):")
print("-"*100)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<12}{'PageRank':<12}{'Composite_Score':<12}")
print("-"*100)

for i in range(20):
    row = df_metrics_sorted.iloc[i]
    print(f"{i+1:<6}{row['Gene']:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<12.4f}"
          f"{row['PageRank']:<12.6f}"
          f"{row['Composite_Score']:<12.2f}")

# ==================== 5. Statistical Analysis to Determine Hub Gene Threshold ====================
print("\n" + "="*70)
print("Phase 4: Statistical Analysis to Determine Hub Gene Threshold")
print("="*70)

# Calculate statistical characteristics of composite scores
composite_scores = df_metrics_sorted['Composite_Score'].values
mean_score = np.mean(composite_scores)
std_score = np.std(composite_scores)
median_score = np.median(composite_scores)
q3_score = np.percentile(composite_scores, 75)

print(f"Composite score statistics:")
print(f"  Mean: {mean_score:.2f}")
print(f"  Standard deviation: {std_score:.2f}")
print(f"  Median: {median_score:.2f}")
print(f"  Third quartile (Q3): {q3_score:.2f}")

# Determine threshold
threshold_1 = mean_score + std_score
threshold_2 = q3_score
threshold = max(threshold_1, threshold_2)

print(f"\nScreening threshold: {threshold:.2f} (Composite score > {threshold:.2f})")

# Screen hub genes
core_genes_df = df_metrics_sorted[df_metrics_sorted['Composite_Score'] > threshold].copy()
core_genes = core_genes_df['Gene'].tolist()

print(f"\nNumber of hub genes identified by statistical threshold: {len(core_genes)}")

# Determine Top 10
if len(core_genes) > 10:
    top10_core_genes = core_genes[:10]
    print(f"More than 10 hub genes identified, selecting top 10 by composite score")
elif len(core_genes) < 10:
    top10_core_genes = df_metrics_sorted.head(10)['Gene'].tolist()
    print(f"Fewer than 10 hub genes identified, selecting top 10 by composite score")
else:
    top10_core_genes = core_genes

print(f"\n🏆 Final Screened Top 10 Core Hub Genes:")
print("-"*50)
for i, gene in enumerate(top10_core_genes, 1):
    row = df_metrics_sorted[df_metrics_sorted['Gene'] == gene].iloc[0]
    print(f"  {i}. {gene:<12} (Degree: {int(row['Degree']):3d}, Composite Score: {row['Composite_Score']:.2f})")

# ==================== 6. Extract Core Subnetwork ====================
print("\n" + "="*70)
print("Phase 5: Constructing Core Hub Gene Subnetwork")
print("="*70)

# Extract subnetwork containing hub genes and their direct neighbors
core_nodes = set(top10_core_genes)
for gene in top10_core_genes:
    if gene in G:
        core_nodes.update(G.neighbors(gene))

# Create subnetwork
subG = G.subgraph(core_nodes).copy()

print(f"Core subnetwork statistics:")
print(f"  Number of nodes: {subG.number_of_nodes()}")
print(f"  Number of edges: {subG.number_of_edges()}")
print(f"  Core genes included: {len([n for n in subG.nodes() if n in top10_core_genes])}")

# ==================== 7. Export Results ====================
print("\n" + "="*70)
print("Phase 6: Exporting Analysis Results")
print("="*70)

# Save detailed information of hub genes
core_genes_detail = df_metrics_sorted[df_metrics_sorted['Gene'].isin(top10_core_genes)].copy()
core_genes_detail = core_genes_detail.set_index('Gene').loc[top10_core_genes].reset_index()
core_genes_detail.to_csv('skin_cancer_core_hub_genes.csv', index=False)
print("✓ Core hub gene details saved: skin_cancer_core_hub_genes.csv")

# Save complete metrics table
df_metrics_sorted.to_csv('skin_cancer_all_genes_metrics.csv', index=False)
print("✓ Complete gene metrics table saved: skin_cancer_all_genes_metrics.csv")

# Save core network
nx.write_graphml(subG, 'skin_cancer_core_network.graphml')
print("✓ Core network saved: skin_cancer_core_network.graphml")

# ==================== 8. Final Summary ====================
print("\n" + "="*70)
print("🎯 Skin Cancer Core Hub Gene Analysis Completed!")
print("="*70)

print("\n📋 Final Screened Top 10 Core Hub Genes (Ranked by Composite Score):")
print("-"*90)
print(f"{'Rank':<6}{'Gene':<12}{'Degree':<8}{'Betweenness':<15}{'PageRank':<15}{'Composite_Score':<12}")
print("-"*90)

for i, gene in enumerate(top10_core_genes, 1):
    row = core_genes_detail[core_genes_detail['Gene'] == gene].iloc[0]
    print(f"{i:<6}{gene:<12}{int(row['Degree']):<8}"
          f"{row['Betweenness_Centrality']:<15.4f}"
          f"{row['PageRank']:<15.6f}"
          f"{row['Composite_Score']:<12.2f}")

print("\n" + "="*70)
print("List of generated files:")
print("  1. skin_cancer_core_hub_genes.csv - Detailed data of core hub genes")
print("  2. skin_cancer_all_genes_metrics.csv - Network metrics for all genes")
print("  3. skin_cancer_core_network.graphml - Cytoscape network file")
print("="*70)